# 02 — Frozen ArrowSpace prior & spectral chart

Build a frozen ArrowSpace prior from a toy feature corpus and visualize the spectral chart.

The prior is built **once** from corpus feature statistics and never updated during training. All components are stored as buffers (zero `nn.Parameter`).

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.build_prior import build_arrow_prior

torch.manual_seed(3407)

## 1. Build the prior from a toy corpus

In [ ]:
F, q, N = 32, 8, 64
embeddings = torch.randn(N, F)
prior = build_arrow_prior(embeddings, q=q, k=4)

print(f"L_F: {prior.L_F.shape}")
print(f"U_q: {prior.U_q.shape}")
print(f"eigvals_q: {prior.eigvals_q.shape}")
print(f"lambdas_ed: {prior.lambdas_ed.shape}")
print(f"lambdas_chart: {prior.lambdas_chart.shape}")
print(f"Trainable parameters: {len(list(prior.parameters()))}")

## 2. Eigenvalue spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(q), prior.eigvals_q.numpy())
ax.set_xlabel("Mode k")
ax.set_ylabel(r"$\nu_k$")
ax.set_title("ArrowSpace Laplacian eigenvalues (smooth modes)")
plt.tight_layout()
plt.savefig("../results/02_eigenvalues.png", dpi=150)
plt.show()

## 3. Projector structure: $\Pi_q = U_q U_q^\top$

In [ ]:
Pi = prior.U_q @ prior.U_q.T
print(f"Pi is symmetric: {torch.allclose(Pi, Pi.T, atol=1e-5)}")
print(f"Pi^2 = Pi: {torch.allclose(Pi @ Pi, Pi, atol=1e-5)}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(Pi.numpy(), cmap="RdBu_r", vmin=-0.5, vmax=0.5)
ax.set_title(r"Projector $\Pi_q = U_q U_q^\top$")
plt.colorbar(im)
plt.tight_layout()
plt.savefig("../results/02_projector.png", dpi=150)
plt.show()

## 4. Band energies and $c_\mathrm{spec}$ for samples

In [ ]:
A = torch.randn(4, F)
e = prior.band_energies(A)
c_spec = prior.chart_energy_descriptor(A)
print(f"Band energies shape: {e.shape}")
print(f"c_spec shape: {c_spec.shape} = 3*q = {3*q}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(4):
    axes[0].bar([x + i*0.2 for x in range(q)], e[i].numpy(), width=0.2, label=f"sample {i}")
axes[0].set_xlabel("Mode k")
axes[0].set_ylabel(r"$e_k$")
axes[0].set_title("Band energies")
axes[0].legend(fontsize=8)

axes[1].imshow(c_spec.numpy(), cmap="viridis", aspect="auto")
axes[1].set_xlabel("Component")
axes[1].set_ylabel("Sample")
axes[1].set_title(r"$c_\mathrm{spec} = [\tilde e, \lambda_\mathrm{chart}, \nu]$")
plt.tight_layout()
plt.savefig("../results/02_c_spec.png", dpi=150)
plt.show()

## 5. Off-manifold energy

In [ ]:
A_raw = torch.randn(4, F)
A_proj = prior.project_to_chart(A_raw)
energy_raw = prior.off_manifold_energy(A_raw)
energy_proj = prior.off_manifold_energy(A_proj)
print(f"Off-manifold energy (raw):      {energy_raw:.4f}")
print(f"Off-manifold energy (projected): {energy_proj:.6f}")